# Setup

In [1]:
import torch

tensors_dir = "../data/tensors"

X_trainval = torch.load(f"{tensors_dir}/X_trainval.pt").float()
X_test = torch.load(f"{tensors_dir}/X_test.pt").float()
y_trainval = torch.load(f"{tensors_dir}/y_trainval.pt").long()
y_test = torch.load(f"{tensors_dir}/y_test.pt").long()
sample_weights = torch.load(f"{tensors_dir}/sample_weights.pt").float()

print(f"X_train: {X_trainval.shape}  {X_trainval.dtype}")
print(f"X_test: {X_test.shape}   {X_test.dtype}")
print(f"y_train: {y_trainval.shape}  {y_trainval.dtype}")
print(f"y_test: {y_test.shape}   {y_test.dtype}")
print(f"weights: {sample_weights.shape}")

X_train: torch.Size([796, 10])  torch.float32
X_test: torch.Size([199, 10])   torch.float32
y_train: torch.Size([796])  torch.int64
y_test: torch.Size([199])   torch.int64
weights: torch.Size([796])


In [2]:
from src import MLP

INPUT_SIZE = X_trainval.shape[1]
NUM_CLASSES = 4

model = MLP(input_size = INPUT_SIZE, hidden_layers=[16, 64, 256, 512, 256, 64, 16],
            output_size = NUM_CLASSES, dropout_p = 0.3, activation='relu')
print(model)

dummy_output = model(torch.randn(8, INPUT_SIZE))
print(f"\noutput shape: {dummy_output.shape}")
print(f"\noutput values: {dummy_output}")

MLP(
  (input): Linear(in_features=10, out_features=16, bias=True)
  (hidden_layers): ModuleList(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): Linear(in_features=64, out_features=256, bias=True)
    (2): Linear(in_features=256, out_features=512, bias=True)
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): Linear(in_features=256, out_features=64, bias=True)
    (5): Linear(in_features=64, out_features=16, bias=True)
  )
  (output): Linear(in_features=16, out_features=4, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (activation): ReLU()
)

output shape: torch.Size([8, 4])

output values: tensor([[ 0.0504, -0.2709,  0.0851,  0.0030],
        [ 0.0276, -0.2589,  0.0984,  0.0140],
        [ 0.0314, -0.2204,  0.0914,  0.0038],
        [ 0.0294, -0.2657,  0.0893,  0.0126],
        [ 0.0140, -0.2569,  0.1067,  0.0216],
        [ 0.0061, -0.2782,  0.1044,  0.0365],
        [-0.0033, -0.2678,  0.1065,  0.0365],
        [ 0.0240, -0.2362,  0.

In [3]:
from src import build_fold_dataloaders

train_loader, val_loader = build_fold_dataloaders(X=X_trainval, y=y_trainval,
                                                  weights=sample_weights, fold_idx=0, batch_size=32, n_splits=5)

X_example_batch, y_example_batch = next(iter(train_loader))
print(f"Test loaders length: \nTrain: {len(train_loader)}   Val: {len(val_loader)}")
print(f"\nX batch info: \nshape: {X_example_batch.shape}   dtype: {X_example_batch.dtype}")
print(f"\nY batch info: \nshape: {y_example_batch.shape}   dtype: {y_example_batch.dtype}")

Test loaders length: 
Train: 20   Val: 5

X batch info: 
shape: torch.Size([32, 10])   dtype: torch.float32

Y batch info: 
shape: torch.Size([32])   dtype: torch.int64


In [4]:
from src import train, _evaluate, _train_one_epoch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_losses = train(model=model, device=device, n_epochs=10, fold_idx=0, train_loader=train_loader, val_loader=val_loader, optimizer=optimizer, criterion=criterion)